# Electrodynamics from Scratch — Lecture 0: Foundations — Experiments to Maxwell

Everything in this course begins with a very simple observation: rubbing certain materials together makes them attract small bits of paper. We say the materials have acquired **electric charge**, an intrinsic property of matter measured in coulombs. Benjamin Franklin assigned the names *positive* and *negative*, and experiments since (most famously Millikan's oil-drop experiment) have shown that charge comes in units of the elementary charge $e = 1.602 \times 10^{-19}\,\mathrm{C}$ and is conserved: it is never created or destroyed, only moved around.

Coulomb's experiment (1785) with a torsion balance established the force law between two point charges $q_1$ and $q_2$ separated by a distance $r$:
$$
\mathbf{F} = \frac{1}{4\pi\varepsilon_0}\,\frac{q_1 q_2}{r^2}\,\hat{\mathbf{r}},
\qquad \frac{1}{4\pi\varepsilon_0} = k_e \approx 8.99 \times 10^9\,\mathrm{N\,m^2/C^2}.
$$
The force is attractive for opposite charges and repulsive for like charges, and it obeys **superposition**: the total force on a charge is the vector sum of the forces from all other charges, each computed as if the others were absent. This linearity is the single most important structural fact of electrostatics — it is what makes every computation in this notebook possible.

Rather than speaking of charges acting on each other at a distance, Faraday and Maxwell taught us to introduce the **electric field**: at every point in space, define
$$
\mathbf{E}(\mathbf{r}) = \frac{\mathbf{F}_{\text{on }q_0}}{q_0}
$$
the force a tiny positive test charge $q_0$ would feel at that point, per unit charge. The field of a point charge is $\mathbf{E} = \frac{1}{4\pi\varepsilon_0}\frac{q}{r^2}\hat{\mathbf{r}}$, and the field of any distribution is the sum of the fields of its parts. In electrostatics the field is curl-free,
$$
\nabla \times \mathbf{E} = \mathbf{0}, \qquad \nabla \cdot \mathbf{E} = \frac{\rho}{\varepsilon_0},
$$
so it can be written as the gradient of a scalar potential, $\mathbf{E} = -\nabla V$. In this notebook we build the field concept step by step: we draw the **field lines** of single charges and of a dipole, render the field as a **vector (quiver) plot**, compute the on-axis field of a charged **ring** and watch it approach the point-charge limit at large distance, and finally examine the exactly solvable fields of a spherical **shell** and a **solid sphere**, which anticipate Gauss's law from Lecture 1.

In [1]:
%matplotlib inline
import numpy as np
import electrodynamics_from_scratch as eds

## 1. The field of a single point charge

A charge $q = +1\,\mathrm{nC}$ sits at the origin. We evaluate $\mathbf{E}$ with `eds.point_charge_field` on a 2D grid in the $xy$-plane (the third coordinate is just $z=0$) and draw the **field lines** with `eds.plot_field_lines`. Field lines point away from a positive charge, their density (lines crossing per unit area perpendicular to the lines) encodes the field strength, and they never cross — exactly the geometric statement of $\nabla \cdot \mathbf{E} = \rho/\varepsilon_0$ with $\nabla \times \mathbf{E} = \mathbf{0}$.

In [ ]:
q = 1e-9  # C
pos = np.array([0.0, 0.0, 0.0])

# field_func(x, y) -> (Ex, Ey) 2D grids, as the plotting helpers expect.
# The plotting mesh contains the charge's exact location, where the Coulomb
# field diverges; we nudge that one point a hair off the charge.
def single_charge_field(x, y):
    X, Y = np.meshgrid(x, y)
    points = np.stack([X.ravel(), Y.ravel(), np.zeros(X.size)], axis=1)
    near = np.linalg.norm(points - pos, axis=1) < 1e-9
    safe = points.copy()
    safe[near] = pos + np.array([1e-6, 0.0, 0.0])
    E = eds.point_charge_field(q, pos, safe)
    return E[:, 0].reshape(X.shape), E[:, 1].reshape(X.shape)

fig, ax = eds.plot_field_lines(single_charge_field, extent=2.0, density=1.4,
                              title='Field lines of a positive point charge')
fig, ax

## 2. Superposition: the electric dipole

Now place $+q$ at $z = +0.25\,\mathrm{m}$ and $-q$ at $z = -0.25\,\mathrm{m}$, so the dipole moment is $\mathbf{p} = q\,\mathbf{d}$ pointing from $-q$ to $+q$. By superposition the total field is the vector sum of the two Coulomb fields, which is exactly what `eds.superpose` computes. Far from the dipole ($r \gg d$) the $1/r^2$ pieces nearly cancel and the leading field falls as $1/r^3$ — this cancellation is the seed of the multipole expansion of Lecture 5.

In [ ]:
d = 0.25
charges = [(q, np.array([0.0, 0.0,  d])),
           (-q, np.array([0.0, 0.0, -d]))]

def dipole_field(x, y):
    X, Y = np.meshgrid(x, y)
    points = np.stack([X.ravel(), Y.ravel(), np.zeros(X.size)], axis=1)
    E, V = eds.superpose(charges, points)
    return E[:, 0].reshape(X.shape), E[:, 1].reshape(X.shape)

fig, ax = eds.plot_field_lines(dipole_field, extent=1.0, density=1.4,
                              title='Field lines of a dipole (+z on top)')
fig, ax

## 3. The dipole field as a vector map

Field lines show direction elegantly but hide the magnitude of the field. A quiver plot from `eds.plot_vector_field` draws an arrow at each grid point whose length tracks $|\mathbf{E}|$: notice how the arrows are long near the charges and short far away, and how along the equatorial plane the field points *opposite* to $\mathbf{p}$ while on the axis it points along $\mathbf{p}$ — the classic $\mathbf{E}_{\text{axis}} = \frac{1}{4\pi\varepsilon_0}\frac{2p}{r^3}\hat{\mathbf{p}}$ versus $\mathbf{E}_{\text{eq}} = -\frac{1}{4\pi\varepsilon_0}\frac{\mathbf{p}}{r^3}$ signatures of a dipole.

In [ ]:
fig, ax = eds.plot_vector_field(dipole_field, extent=1.0, n=15,
                               title='Dipole field arrows (length ~ |E|)')
fig, ax

## 4. Continuous charge: on-axis field of a charged ring

Superposition extends from discrete charges to **continuous distributions** by integration. A ring of radius $R = 0.5\,\mathrm{m}$ carrying total charge $Q = 2\,\mathrm{nC}$ in the $xy$-plane has, on its axis at distance $z$, the exact result
$$
E_z(z) = \frac{1}{4\pi\varepsilon_0}\,\frac{Q\,z}{(z^2 + R^2)^{3/2}},
$$
obtained by integrating around the ring (the transverse components cancel by symmetry). `eds.ring_field` does the integral numerically. At the center the field is **zero** — every element's contribution is cancelled by the element diametrically opposite — and for $z \gg R$ the ring looks like a point charge, $E_z \to k_e Q/z^2$.

In [ ]:
Q = 2e-9
R = 0.5
center = np.array([0.0, 0.0, 0.0])

z = np.linspace(0.01, 4.0, 200)
points = np.stack([np.zeros_like(z), np.zeros_like(z), z], axis=1)

E_ring = eds.ring_field(Q, R, center, points, n=400)
Ez_numeric = E_ring[:, 2]

k_e = 1.0 / (4.0 * np.pi * eds.EPSILON_0)
Ez_exact = k_e * Q * z / (z**2 + R**2) ** 1.5       # closed form
Ez_point  = k_e * Q / z**2                          # point-charge limit

fig, ax = eds.plot_decay(z, {'ring (numeric)': Ez_numeric,
                             'ring (exact)': Ez_exact,
                             'point charge Q/z^2': Ez_point},
                        loglog=True,
                        title='On-axis field of a charged ring vs the point-charge limit')
ax.set_xlabel('axial distance z (m)')
ax.set_ylabel('E_z (V/m)')
fig, ax

## 5. The spherical shell: field hidden and revealed

A thin spherical shell of radius $R = 1\,\mathrm{m}$ with total charge $Q$ is the first exactly solvable continuous distribution: **everywhere inside the shell the field vanishes**, while outside the shell acts as if all its charge were concentrated at the center. `eds.shell_field` returns this exact solution. We sample $\mathbf{E}$ along a radial line and compare $|\mathbf{E}|$ with the Coulomb form $k_e Q / r^2$: the two coincide for $r > R$ and the field drops to zero for $r < R$. This cancellation is a theorem (shell theorem / Gauss's law), not an accident.

In [ ]:
Q = 2e-9
R_shell = 1.0

r = np.linspace(0.05, 4.0, 200)
points = np.stack([r, np.zeros_like(r), np.zeros_like(r)], axis=1)

E_shell = eds.shell_field(Q, R_shell, points)
Emag_shell = np.linalg.norm(E_shell, axis=1)

fig, ax = eds.plot_decay(r, {'|E| shell': Emag_shell,
                             'k_e Q / r^2': k_e * Q / r**2},
                        loglog=True,
                        title='Field of a charged shell: zero inside, Coulomb outside')
ax.axvline(R_shell, color='gray', linestyle='--', linewidth=1)
ax.text(R_shell * 1.05, Emag_shell.max(), ' r = R', color='gray')
ax.set_xlabel('r (m)')
ax.set_ylabel('|E| (V/m)')
fig, ax

## 6. The solid sphere: the field grows linearly inside

Fill the shell in: a uniformly charged **solid sphere** of radius $R$ and total charge $Q$. Outside, the shell theorem again gives the pure Coulomb field. Inside, only the charge interior to radius $r$ contributes, and since that enclosed charge scales as $r^3$ while the distance scales as $r^2$, the field grows **linearly**:
$$
|E(r)| = \frac{1}{4\pi\varepsilon_0}\,\frac{Q\,r}{R^3} \quad (r < R).
$$
`eds.solid_sphere_field` returns this exact solution — compare the computed magnitude against both asymptotic forms below.

In [ ]:
R_sphere = 1.0

E_sphere = eds.solid_sphere_field(Q, R_sphere, points)
Emag_sphere = np.linalg.norm(E_sphere, axis=1)

E_inside = k_e * Q * r / R_sphere**3      # r < R
E_outside = np.where(r > R_sphere, k_e * Q / r**2, np.nan)

fig, ax = eds.plot_decay(r, {'|E| solid sphere': Emag_sphere,
                             'k_e Q r / R^3 (inside)': E_inside,
                             'k_e Q / r^2 (outside)': E_outside},
                        loglog=True,
                        title='Field of a uniformly charged solid sphere')
ax.axvline(R_sphere, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('r (m)')
ax.set_ylabel('|E| (V/m)')
fig, ax

## Exercises

1. **Conceptual — superposition.** Two charges $+q$ and $+3q$ are separated by a distance $d$. Using the fact that electric fields add vectorially, argue (without computing) where on the line between them the total field vanishes. Then verify your answer numerically: evaluate `eds.superpose` with `charges=[(q, ...), (3*q, ...)]` on a 1D line of points between the charges and find where $|\mathbf{E}|$ is smallest.

2. **Coding — equipotentials.** Build a `potential_func(x, y)` for the dipole of Section 2 using `eds.point_charge_potential` (sum the two contributions, flattening the meshgrid as in the worked cells) and plot the equipotential surfaces with `eds.plot_equipotentials(potential_func, extent=1.0, levels=20)`. Near which point do the equipotential contours crowd together most, and what does that tell you about $|\mathbf{E}| = -\nabla V$ there?

3. **Coding — ring vs shell.** Using `eds.ring_field` with points placed along the $x$-axis in the plane of the ring (the transverse plane, not the axis), show numerically that the field in the plane is *not* zero for $x < R$, unlike the interior of the shell in Section 5. Explain in words why symmetry protects the interior of a spherical shell but not the interior of a ring.

4. **Conceptual — shell theorem.** In Section 5 the computed shell field is identically zero for $r < R$ and matches $k_e Q/r^2$ for $r > R$. Explain how this behavior foreshadows Gauss's law, $\oint \mathbf{E} \cdot d\mathbf{a} = Q_{\text{enc}}/\varepsilon_0$: what does a spherical Gaussian surface inside the shell enclose, and what does one outside enclose?

5. **Coding — line charge.** A thin wire of length $2\,\mathrm{m}$ along the $z$-axis carries total charge $Q = 5\,\mathrm{nC}$. Use `eds.line_charge_field(charge, p0, p1, points)` with `p0 = [0, 0, -1]`, `p1 = [0, 0, 1]` to compute $\mathbf{E}$ at points on the $x$-axis from $0.1$ to $5\,\mathrm{m}$. Plot $|\mathbf{E}|$ on a log-log plot (e.g. with `eds.plot_decay`) and identify the two power-law regimes: $\sim 1/x$ when close to the wire compared with its length, and $\sim 1/x^2$ when far away. (You can measure exponents with `eds.power_law_fit`.)

6. **Conceptual — field lines.** In Section 1 the field lines of a point charge spread out radially. Explain why field lines can never cross, and what property of $\nabla \times \mathbf{E} = \mathbf{0}$ guarantees that (in electrostatics) field lines never form closed loops. What would have to change in a time-varying situation for closed electric field lines to become possible?